# Entendiendo Derivation Paths (Rutas de Derivación)

¿Alguna vez te has preguntado cómo una sola frase de 12 palabras (Mnemonic) puede generar miles de direcciones diferentes para Bitcoin, Ethereum, Solana y más? 

La respuesta está en los **Derivation Paths**. En este notebook exploraremos cómo funcionan bajo el estándar **BIP-44**.

### 1. Instalación de Dependencias
Utilizaremos `bip_utils`, una de las librerías más potentes para manejar derivaciones jerárquicas determínisticas.

In [5]:
!pip install bip-utils mnemonic

### 2. Anatomía de un Derivation Path

Una ruta típica se ve así: `m / purpose' / coin_type' / account' / change / address_index` 

- **m**: La clave maestra generada a partir de la frase semilla.
- **purpose**: El estándar de derivación (ej. 44 para Legacy, 84 para Native SegWit).
- **coin_type**: El ID de la moneda (60 para ETH, 0 para BTC, 501 para SOL).
- **account**: Permite tener múltiples "sub-billeteras" (ej. ahorros vs gastos).
- **change**: 0 para direcciones de recepción, 1 para cambio (interno).
- **address_index**: El número de cuenta secuencial (Cuenta 1, Cuenta 2, etc.).

--- 
### 3. Un Mnemonic, Múltiples Redes
Vamos a generar una frase semilla y ver cómo derivamos direcciones para diferentes redes.

In [7]:
from bip_utils import Bip39SeedGenerator, Bip44, Bip44Coins, Bip44Changes, Bip84, Bip84Coins
from mnemonic import Mnemonic

# 1. Generar Mnemonic
mnemonic = "test test test test test test test test test test test  junk"
print(f"📦 Frase Semilla: {mnemonic}\n")

# 2. Generar Seed binario a partir del mnemonic
seed_bytes = Bip39SeedGenerator(mnemonic).Generate()

def print_derivation(name, path, address):
    print(f"🌐 Red: {name:<15} | 🛤️ Ruta: {path:<18} | 💳 Address: {address}")

print("--- DERIVACIONES BIP-44 ---")

# Ethereum
eth_ctx = Bip44.FromSeed(seed_bytes, Bip44Coins.ETHEREUM).Purpose().Coin().Account(0).Change(Bip44Changes.CHAIN_EXT).AddressIndex(0)
print_derivation("Ethereum", "m/44'/60'/0'/0/0", eth_ctx.PublicKey().ToAddress())

# Bitcoin Legacy
btc_ctx = Bip44.FromSeed(seed_bytes, Bip44Coins.BITCOIN).Purpose().Coin().Account(0).Change(Bip44Changes.CHAIN_EXT).AddressIndex(0)
print_derivation("Bitcoin (1...)", "m/44'/0'/0'/0/0", btc_ctx.PublicKey().ToAddress())

# Solana
sol_ctx = Bip44.FromSeed(seed_bytes, Bip44Coins.SOLANA).Purpose().Coin().Account(0).Change(Bip44Changes.CHAIN_EXT).AddressIndex(0)
print_derivation("Solana", "m/44'/501'/0'/0/0", sol_ctx.PublicKey().ToAddress())

📦 Frase Semilla: test test test test test test test test test test test  junk

--- DERIVACIONES BIP-44 ---
🌐 Red: Ethereum        | 🛤️ Ruta: m/44'/60'/0'/0/0   | 💳 Address: 0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266
🌐 Red: Bitcoin (1...)  | 🛤️ Ruta: m/44'/0'/0'/0/0    | 💳 Address: 1Ei9UmLQv4o4UJTy5r5mnGFeC9auM3W5P1
🌐 Red: Solana          | 🛤️ Ruta: m/44'/501'/0'/0/0  | 💳 Address: BJo25z6yW1fZPofStPeA4KXRJrKoKtmdqkFxXU6yG2oG


### 4. Bitcoin y los Estándares Modernos (BIP-84)

Bitcoin ha evolucionado. Si usas una ruta que empieza por `84'`, generarás direcciones **Native SegWit** (Bech32), que son más baratas de transaccionar.

In [8]:
# Bitcoin Native SegWit (bech32 - bc1...)
btc_segwit = Bip84.FromSeed(seed_bytes, Bip84Coins.BITCOIN).Purpose().Coin().Account(0).Change(Bip44Changes.CHAIN_EXT).AddressIndex(0)
print_derivation("BTC Native SegWit", "m/84'/0'/0'/0/0", btc_segwit.PublicKey().ToAddress())

🌐 Red: BTC Native SegWit | 🛤️ Ruta: m/84'/0'/0'/0/0    | 💳 Address: bc1q4qw42stdzjqs59xvlrlxr8526e3nunw7mp73te


### 5. Laboratorio Interactivo: El Índice de Cuenta

Cuando haces clic en "Crear nueva cuenta" en MetaMask o Phantom, el software simplemente incrementa el último número de la ruta (`address_index`). 

Ejecuta la siguiente celda para generar las primeras 5 cuentas de Ethereum a partir de la misma frase semilla.

In [9]:
print(f"Cuentas derivadas del Mnemonic para Ethereum:\n")
for i in range(5):
    account = Bip44.FromSeed(seed_bytes, Bip44Coins.ETHEREUM).Purpose().Coin().Account(0).Change(Bip44Changes.CHAIN_EXT).AddressIndex(i)
    path = f"m/44'/60'/0'/0/{i}"
    print(f"Cuenta #{i+1} | Ruta: {path} | Address: {account.PublicKey().ToAddress()}")

Cuentas derivadas del Mnemonic para Ethereum:

Cuenta #1 | Ruta: m/44'/60'/0'/0/0 | Address: 0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266
Cuenta #2 | Ruta: m/44'/60'/0'/0/1 | Address: 0x70997970C51812dc3A010C7d01b50e0d17dc79C8
Cuenta #3 | Ruta: m/44'/60'/0'/0/2 | Address: 0x3C44CdDdB6a900fa2b585dd299e03d12FA4293BC
Cuenta #4 | Ruta: m/44'/60'/0'/0/3 | Address: 0x90F79bf6EB2c4f870365E785982E1f101E93b906
Cuenta #5 | Ruta: m/44'/60'/0'/0/4 | Address: 0x15d34AAf54267DB7D7c367839AAf71A00a2C6A65


### Resumen de Rutas Comunes

| Red | Propósito | Ruta Estándar |
| :--- | :--- | :--- |
| **Bitcoin Legacy** | BIP-44 | `m/44'/0'/0'/0/i` |
| **Bitcoin SegWit** | BIP-49 | `m/49'/0'/0'/0/i` |
| **Bitcoin Native** | BIP-84 | `m/84'/0'/0'/0/i` |
| **Ethereum** | BIP-44 | `m/44'/60'/0'/0/i` |
| **Solana** | BIP-44 | `m/44'/501'/0'/0/i` |

--- 
**⚠️ Importante:** Si intentas restaurar una billetera y no ves tus fondos, es probable que el software esté usando un **Derivation Path** diferente al que usaste originalmente.